In [1]:
import sys
sys.path.insert(0, '../lib')

import json
import glob
import os
import pickle
import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd

import common_data

In [2]:
%config InlineBackend.figure_format = "retina"

In [3]:
pd.options.display.max_columns = 300
pd.options.display.max_rows = 350
pd.options.display.max_colwidth = 10000

In [4]:
mpl.rc_file_defaults()

In [5]:
ehr_labels = pd.read_csv(common_data.CLINICAL_LABELS, index_col=0)
ehr = pd.read_csv(common_data.CLINICAL, index_col=0)

How many BALs?

In [6]:
ehr.BAL_performed.value_counts()

False    13546
True      1741
Name: BAL_performed, dtype: int64

How many patients had BALs?

In [7]:
ehr.patient[ehr.BAL_performed].nunique()

690

In [8]:
labels_cols = ehr_labels.columns[~ehr_labels.columns.isin(ehr.columns)]

In [9]:
ehr = ehr.merge(ehr_labels[labels_cols], left_index=True, right_index=True)

BAL pathogen breakdown

In [10]:
ehr.pathogen_groups[ehr.BAL_performed].value_counts()

pathogen-negative                                            592
SARS-CoV-2                                                   242
Gram+                                                        198
Gram-*                                                       181
SARS-CoV-2; Gram+                                             98
Gram-*; Gram+                                                 82
Pseudomonas aeruginosa                                        81
Other viruses                                                 68
Gram-*; SARS-CoV-2                                            41
Gram-*; Pseudomonas aeruginosa                                29
Gram-*; SARS-CoV-2; Gram+                                     27
Pseudomonas aeruginosa; Gram+                                 20
Pseudomonas aeruginosa; SARS-CoV-2                            14
Other viruses; Gram+                                          14
Gram-*; Other viruses                                          8
Gram-*; Other viruses; Gr

How many pneumonia episodes?

In [11]:
ehr.Episode_category.notna().sum()

928

How many BALs starting pneumonia or NPC episodes?

In [23]:
IDX_TOTAL = ehr.Episode_category.notna() & ehr.BAL_performed

In [24]:
IDX_TOTAL.sum()

927

For how many patients?

In [25]:
ehr.patient[IDX_TOTAL].nunique()

687

Breakdown of BALs starting episodes by pathogen

In [26]:
ehr.pathogen_groups[IDX_TOTAL].value_counts(normalize=True)

pathogen-negative                                            0.323944
SARS-CoV-2                                                   0.128927
Gram+                                                        0.112676
Gram-*                                                       0.097508
SARS-CoV-2; Gram+                                            0.074756
Gram-*; Gram+                                                0.054171
Other viruses                                                0.046587
Pseudomonas aeruginosa                                       0.037920
Gram-*; SARS-CoV-2                                           0.028169
Gram-*; SARS-CoV-2; Gram+                                    0.018418
Gram-*; Pseudomonas aeruginosa                               0.016251
Pseudomonas aeruginosa; Gram+                                0.014085
Pseudomonas aeruginosa; SARS-CoV-2                           0.008667
Gram-*; Other viruses; Gram+                                 0.006501
Other viruses; Gram+

In [27]:
ehr.pathogen_groups[IDX_TOTAL].value_counts()

pathogen-negative                                            299
SARS-CoV-2                                                   119
Gram+                                                        104
Gram-*                                                        90
SARS-CoV-2; Gram+                                             69
Gram-*; Gram+                                                 50
Other viruses                                                 43
Pseudomonas aeruginosa                                        35
Gram-*; SARS-CoV-2                                            26
Gram-*; SARS-CoV-2; Gram+                                     17
Gram-*; Pseudomonas aeruginosa                                15
Pseudomonas aeruginosa; Gram+                                 13
Pseudomonas aeruginosa; SARS-CoV-2                             8
Gram-*; Other viruses; Gram+                                   6
Other viruses; Gram+                                           6
Gram-*; Pseudomonas aerug

How many had some pathogen and started a pneumonia episode?

In [28]:
IDX_PNEU_PATH = IDX_TOTAL & (
    ehr.Episode_category.ne('NPC')
    & ehr.pathogen_groups.ne('pathogen-negative')
)

In [29]:
IDX_PNEU_PATH.sum()

619

In [37]:
ehr.patient[IDX_PNEU_PATH].nunique()

452

In [30]:
IDX_PNEU_PATH.sum() / IDX_TOTAL.sum()

0.6677454153182308

How many samples were adjudicated as pneumonia but without a pathogen?

In [31]:
IDX_PNEU_NO_PATH = IDX_TOTAL & (
    ehr.Episode_category.ne('NPC')
    & ehr.pathogen_groups.eq('pathogen-negative')
)

In [32]:
IDX_PNEU_NO_PATH.sum()

194

In [33]:
IDX_PNEU_NO_PATH.sum() / IDX_TOTAL.sum()

0.209277238403452

How many samples were NPCs?

In [34]:
IDX_NPC = IDX_TOTAL & ehr.Episode_category.eq('NPC')

In [35]:
IDX_NPC.sum()

114

In [36]:
IDX_NPC.sum() / IDX_TOTAL.sum()

0.12297734627831715

Fraction of patients with pneumonia without pathogen

In [45]:
pts_with_pneumonia_path = set(ehr.patient[IDX_PNEU_PATH].unique())

In [46]:
pts_with_pneumonia_no_path = set(ehr.patient[IDX_PNEU_NO_PATH].unique())

In [47]:
len(pts_with_pneumonia_no_path) / len(pts_with_pneumonia_path | pts_with_pneumonia_no_path)

0.3102866779089376